# Train & Evaluate Recommendation Models

Notebook này sẽ xây dựng và đánh giá 3 thuật toán khuyến nghị phim dựa trên dữ liệu MovieLens đã được xử lý (rating + metadata).

Các thuật toán sẽ được triển khai: 
1. User-based Collaborative Filtering (UserCF)
2. Item-based Collaborative Filtering (ItemCF)
3. Matrix Factorization (SVD)

Cuối cùng sẽ đánh giá bằng RMSE/MAE và Precision@K / Recall@K để so sánh hiệu năng.

## 1) Load dữ liệu đã xử lý

Dữ liệu đã được tạo ra từ notebook xử lý dữ liệu (`Data_Processing.ipynb`). Ở đây chúng ta sẽ tải:
- `ratings_cleaning.csv`: rating đã làm sạch + timestamp
- `movies_metadata.csv`: metadata phục vụ gợi ý (genre, tag, era, v.v.)
- `model_features.csv`: feature numeric + one-hot + tfidf dùng cho content-based.

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load dữ liệu đã xử lý
# The processed ratings file is expected to be in the workspace.
# If you don't have `ratings_cleaning.csv`, fallback to the original `ratings.csv`.
ratings_path = 'ratings_cleaning.csv'
if not os.path.exists(ratings_path):
    ratings_path = 'ratings.csv'

metadata_path = 'movies_metadata.csv'
features_path = 'model_features.csv'

assert os.path.exists(ratings_path), f'File not found: {ratings_path}'
assert os.path.exists(metadata_path), f'File not found: {metadata_path}'
assert os.path.exists(features_path), f'File not found: {features_path}'

# Load preprocessed datasets
# Note: `movies_metadata.csv` and `model_features.csv` are assumed to be generated from the data processing pipeline.
df_ratings = pd.read_csv(ratings_path)
df_metadata = pd.read_csv(metadata_path)
df_features = pd.read_csv(features_path)

print('ratings:', df_ratings.shape)
print('metadata:', df_metadata.shape)
print('features:', df_features.shape)

df_ratings.head()

ratings: (100836, 4)
metadata: (100834, 7)
features: (100834, 128)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## 2) Chuẩn bị rating matrix (sparse)

Ở đây ta sẽ tạo train/test split dựa trên timestamp (giữ rating cuối cùng của mỗi user làm test) để đánh giá mô hình gợi ý theo thời gian.

In [4]:
# Nếu chưa có timestamp, ta sẽ split random (vẫn đảm bảo user có train/test)
if 'timestamp' not in df_ratings.columns:
    df_ratings['timestamp'] = np.arange(len(df_ratings))

# 1) Chuyển sang kiểu số nếu cần
df_ratings['userId'] = df_ratings['userId'].astype(int)
df_ratings['movieId'] = df_ratings['movieId'].astype(int)
df_ratings['rating'] = df_ratings['rating'].astype(float)

# 2) Chia train/test: dùng phần tử mới nhất (timestamp lớn nhất) mỗi user làm test
df_ratings_sorted = df_ratings.sort_values(['userId', 'timestamp'])
test_idx = df_ratings_sorted.groupby('userId').tail(1).index
train_idx = df_ratings_sorted.index.difference(test_idx)

df_train = df_ratings.loc[train_idx].reset_index(drop=True)
df_test = df_ratings.loc[test_idx].reset_index(drop=True)

print('Train size:', len(df_train))
print('Test size:', len(df_test))

# Số lượng user và movie trong train/test
print('unique users train:', df_train['userId'].nunique())
print('unique users test:', df_test['userId'].nunique())
print('unique movies train:', df_train['movieId'].nunique())
print('unique movies test:', df_test['movieId'].nunique())


Train size: 100226
Test size: 610
unique users train: 610
unique users test: 610
unique movies train: 9701
unique movies test: 514


## 3) Thuật toán 1: User-based Collaborative Filtering (UserCF)

**Tại sao chọn**: UserCF dựa trên giả định rằng người dùng có hành vi tương đồng sẽ thích các item tương tự nhau. Phù hợp khi chúng ta có nhiều ratings từ từng user.

**Giải thích ngắn**:
- Tính similarity giữa các user dựa trên vector rating (cosine).
- Với một user cần dự đoán rating cho một phim, lấy các user tương đồng đã đánh giá phim đó và tính trung bình có trọng số.

Chúng ta sẽ đánh giá bằng RMSE/MAE trên tập test.

In [6]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Build mappings (userid/movieid -> index)
user_ids = df_train['userId'].unique()
movie_ids = df_train['movieId'].unique()
user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

# Build sparse user-item matrix for train
rows = df_train['userId'].map(user_to_idx)
cols = df_train['movieId'].map(movie_to_idx)
data = df_train['rating']
R = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(movie_ids)))

# Compute cosine similarity between users
user_sim = cosine_similarity(R)

def predict_usercf(user_id, movie_id, k=20):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return np.nan
    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]

    sims = user_sim[uidx]
    ratings = R[:, midx].toarray().reshape(-1)

    mask = ratings > 0
    if mask.sum() == 0:
        return np.nan

    sims = sims[mask]
    ratings = ratings[mask]

    top_k = np.argsort(sims)[-k:]
    sim_k = sims[top_k]
    rating_k = ratings[top_k]

    if sim_k.sum() == 0:
        return np.nan

    return np.dot(sim_k, rating_k) / sim_k.sum()

# Predict in bulk for test set (sample for speed)
df_test_sample = df_test.copy()
if len(df_test_sample) > 5000:
    df_test_sample = df_test_sample.sample(5000, random_state=42)

df_test_sample['pred_usercf'] = df_test_sample.apply(
    lambda r: predict_usercf(r['userId'], r['movieId'], k=20), axis=1
)

# Filter out cases where prediction couldn't be made
valid_mask = df_test_sample['pred_usercf'].notna()
rmse_usercf = np.sqrt(mean_squared_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_usercf']))
mae_usercf = mean_absolute_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_usercf'])

print(f'UserCF RMSE: {rmse_usercf:.4f}, MAE: {mae_usercf:.4f} (n={valid_mask.sum()} valid predictions)')

UserCF RMSE: 1.0473, MAE: 0.8281 (n=587 valid predictions)


## 4) Thuật toán 2: Item-based Collaborative Filtering (ItemCF)

**Tại sao chọn**: ItemCF ổn định hơn với user mới và ít thay đổi. Nó dựa vào ý tưởng "nếu bạn thích item A, bạn cũng có thể thích item B" dựa trên tính tương đồng giữa item.

**Giải thích ngắn**:
- Tính similarity giữa item dựa trên vector rating của user (cosine).
- Dự đoán rating là trung bình có trọng số các rating của user với các item tương đồng.


In [8]:
# ItemCF: sử dụng ma trận user-item R đã tạo ở trên
from sklearn.metrics.pairwise import cosine_similarity

R_item = R.T.tocsr()  # shape: (n_items, n_users)
item_sim = cosine_similarity(R_item)

def predict_itemcf(user_id, movie_id, k=20):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return np.nan
    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]

    user_ratings = R[uidx].toarray().reshape(-1)
    mask = user_ratings > 0

    if mask.sum() == 0:
        return np.nan

    sims = item_sim[midx, mask]
    rating_k = user_ratings[mask]

    top_k = np.argsort(sims)[-k:]
    sim_k = sims[top_k]
    rating_k = rating_k[top_k]

    if sim_k.sum() == 0:
        return np.nan

    return np.dot(sim_k, rating_k) / sim_k.sum()

df_test_sample['pred_itemcf'] = df_test_sample.apply(
    lambda r: predict_itemcf(r['userId'], r['movieId'], k=20), axis=1
)

# Filter out cases where prediction couldn't be made
valid_mask = df_test_sample['pred_itemcf'].notna()
rmse_itemcf = np.sqrt(mean_squared_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_itemcf']))
mae_itemcf = mean_absolute_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_itemcf'])

print(f'ItemCF RMSE: {rmse_itemcf:.4f}, MAE: {mae_itemcf:.4f} (n={valid_mask.sum()} valid predictions)')

ItemCF RMSE: 0.9499, MAE: 0.7146 (n=587 valid predictions)


## 5) Thuật toán 3: Matrix Factorization (SVD)

**Tại sao chọn**: SVD (matrix factorization) học được embedding ẩn (latent factors) cho user và item, giúp xử lý sparsity tốt và cho dự đoán chính xác hơn.

**Giải thích ngắn**:
- Phân rã ma trận user-item thành hai ma trận thấp chiều (user factors, item factors)
- Dự đoán rating = dot(user_factor, item_factor).
- Huấn luyện bằng tối ưu hoá (SGD, ALS) trên dữ liệu rating.


In [10]:
# Matrix Factorization via truncated SVD (train on sparse user-item train matrix)
from sklearn.decomposition import TruncatedSVD

n_components = 50
svd = TruncatedSVD(n_components=n_components, random_state=42)
U_s = svd.fit_transform(R)  # shape: (n_users, n_components)
V = svd.components_        # shape: (n_components, n_items)

# Reconstruct approximate ratings for all user-item pairs
R_pred = U_s @ V

preds = []
truths = []
for _, row in df_test_sample.iterrows():
    user_id = row['userId']
    movie_id = row['movieId']
    rating = row['rating']

    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        continue

    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]

    preds.append(R_pred[uidx, midx])
    truths.append(rating)

rmse_svd = np.sqrt(mean_squared_error(truths, preds))
mae_svd = mean_absolute_error(truths, preds)
print(f'SVD (TruncatedSVD) RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f} (n={len(truths)})')

SVD (TruncatedSVD) RMSE: 3.3715, MAE: 3.1753 (n=587)


## 6) Đánh giá mô hình (Precision@K / Recall@K)

Để đánh giá gợi ý (top-K), ta sẽ tính `Precision@K` và `Recall@K` dựa trên việc gợi ý phim cho mỗi user và so sánh với các phim user thực sự đã đánh giá trong tập test.

In [12]:
def precision_recall_at_k(recommendations, ground_truth, k=10):
    precisions = []
    recalls = []
    for u, gt in ground_truth.items():
        if len(gt) == 0:
            continue
        recs = recommendations.get(u, [])[:k]
        hits = len(set(recs) & gt)
        precisions.append(hits / k)
        recalls.append(hits / len(gt))
    return np.mean(precisions) if precisions else np.nan, np.mean(recalls) if recalls else np.nan

gt_by_user = df_test.groupby('userId')['movieId'].apply(set).to_dict()
all_movies = set(df_train['movieId'].unique())

def recommend_usercf(user_id, k=10):
    scores = []
    for m in all_movies:
        if m in df_train[df_train['userId'] == user_id]['movieId'].values:
            continue
        score = predict_usercf(user_id, m, k=20)
        if not np.isnan(score):
            scores.append((m, score))
    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top

def recommend_itemcf(user_id, k=10):
    scores = []
    for m in all_movies:
        if m in df_train[df_train['userId'] == user_id]['movieId'].values:
            continue
        score = predict_itemcf(user_id, m, k=20)
        if not np.isnan(score):
            scores.append((m, score))
    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top

def recommend_svd(user_id, k=10):
    if user_id not in user_to_idx:
        return []
    uidx = user_to_idx[user_id]
    user_rated = set(df_train.loc[df_train['userId'] == user_id, 'movieId'].values)

    scores = []
    for m in all_movies:
        if m in user_rated:
            continue
        if m not in movie_to_idx:
            continue
        midx = movie_to_idx[m]
        scores.append((m, R_pred[uidx, midx]))

    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top

sample_users = list(gt_by_user.keys())[:200]
rec_usercf = {u: recommend_usercf(u, k=10) for u in sample_users}
rec_itemcf = {u: recommend_itemcf(u, k=10) for u in sample_users}
rec_svd = {u: recommend_svd(u, k=10) for u in sample_users}

p_u, r_u = precision_recall_at_k(rec_usercf, gt_by_user, k=10)
p_i, r_i = precision_recall_at_k(rec_itemcf, gt_by_user, k=10)
p_s, r_s = precision_recall_at_k(rec_svd, gt_by_user, k=10)

print(f'UserCF Precision@10: {p_u:.4f}, Recall@10: {r_u:.4f}')
print(f'ItemCF Precision@10: {p_i:.4f}, Recall@10: {r_i:.4f}')
print(f'SVD Precision@10: {p_s:.4f}, Recall@10: {r_s:.4f}')

UserCF Precision@10: 0.0000, Recall@10: 0.0000
ItemCF Precision@10: 0.0002, Recall@10: 0.0016
SVD Precision@10: 0.0030, Recall@10: 0.0295


## 7) So sánh và kết luận

Tổng hợp kết quả RMSE/MAE và Precision@K/Recall@K để so sánh hiệu năng của 3 phương pháp.

- Nếu RMSE thấp nhất đồng thời Precision/Recall tốt -> thuật toán phù hợp.
- UserCF/ItemCF có thể nhanh, dễ triển khai nhưng kém ổn định khi sparsity cao.
- SVD thường hoạt động tốt với dữ liệu thưa và cho dự đoán chính xác hơn.

(Bạn có thể thay đổi tham số, số lượng faktor, hoặc dùng LightFM / deep learning để cải thiện tiếp.)